<a href="https://colab.research.google.com/github/cjEbn/engenharia-de-prompt-fundamentos/blob/main/aula07consulta_de_CEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import re # Importa o módulo 're' para validação de expressões regulares

def validar_cep(cep):
    """
    Valida o formato do CEP.
    Um CEP válido deve conter exatamente 8 dígitos numéricos.

    Args:
        cep (str): A string do CEP a ser validada.

    Returns:
        tuple: (True, "") se o CEP for válido, ou (False, mensagem_de_erro) caso contrário.
    """
    if not isinstance(cep, str):
        return False, "O CEP deve ser uma string."
    # Verifica se o CEP contém exatamente 8 dígitos numéricos
    if not re.fullmatch(r'^\d{8}$', cep):
        return False, "CEP inválido. Deve conter 8 dígitos numéricos (ex: 01001000)."
    return True, ""

def consultar_cep_api(cep):
    """
    Consulta informações de um CEP utilizando a API ViaCEP.

    Args:
        cep (str): O número do CEP a ser consultado (somente dígitos).

    Returns:
        dict: Um dicionário com o status da operação ('success' ou 'error'),
              uma mensagem descritiva e os dados do CEP (se a consulta for bem-sucedida).
    """
    url = f"https://viacep.com.br/ws/{cep}/json/"

    try:
        # Adiciona um timeout para evitar que a requisição fique "pendurada"
        resposta = requests.get(url, timeout=5)
        # Lança um HTTPError para respostas de status 4xx/5xx
        resposta.raise_for_status()

        dados = resposta.json()

        # Verifica se a API retornou um erro (CEP não encontrado)
        if "erro" in dados and dados["erro"]:
            return {"status": "error", "message": "CEP não encontrado ou inválido pela API.", "data": None}
        else:
            return {"status": "success", "message": "Consulta realizada com sucesso!", "data": dados}

    except requests.exceptions.Timeout:
        return {"status": "error", "message": "Tempo limite da requisição excedido. Verifique sua conexão com a internet ou a disponibilidade da API.", "data": None}
    except requests.exceptions.ConnectionError:
        return {"status": "error", "message": "Erro de conexão. Verifique sua internet ou se a URL da API está correta.", "data": None}
    except requests.exceptions.HTTPError as e:
        return {"status": "error", "message": f"Erro HTTP na requisição: {e}. Status code: {e.response.status_code}", "data": None}
    except ValueError: # Erro ao decodificar JSON
        return {"status": "error", "message": "Resposta da API inválida: não foi possível decodificar JSON.", "data": None}
    except Exception as e:
        # Captura qualquer outra exceção inesperada
        return {"status": "error", "message": f"Um erro inesperado ocorreu: {e}", "data": None}

def exibir_resultado(resultado_consulta):
    """
    Exibe o resultado da consulta de CEP de forma estruturada no console.

    Args:
        resultado_consulta (dict): O dicionário de resultado retornado pela função consultar_cep_api.
    """
    if resultado_consulta["status"] == "success":
        dados = resultado_consulta["data"]
        print("\n==============================")
        print("📍 Resultado da Consulta de CEP:")
        print("==============================")
        print(f"  CEP:         {dados.get('cep', 'Não disponível')}")
        print(f"  Logradouro:  {dados.get('logradouro', 'Não disponível')}")
        print(f"  Complemento: {dados.get('complemento', 'Não disponível')}")
        print(f"  Bairro:      {dados.get('bairro', 'Não disponível')}")
        print(f"  Cidade:      {dados.get('localidade', 'Não disponível')}")
        print(f"  Estado (UF): {dados.get('uf', 'Não disponível')}")
        print(f"  IBGE:        {dados.get('ibge', 'Não disponível')}")
        print(f"  GIA:         {dados.get('gia', 'Não disponível')}")
        print(f"  DDD:         {dados.get('ddd', 'Não disponível')}")
        print(f"  SIAFI:       {dados.get('siafi', 'Não disponível')}")
        print("==============================")
    else:
        print("\n==============================")
        print("❌ Erro na Consulta de CEP")
        print("==============================")
        print(f"  Mensagem: {resultado_consulta['message']}")
        print("==============================")

def main():
    """
    Função principal que gerencia a interação com o usuário
    e o fluxo de consulta de CEP.
    """
    print("Bem-vindo ao Consultor de CEP!")
    print("Digite 'sair' a qualquer momento para encerrar o programa.")

    while True:
        cep_input = input("\nDigite o CEP (somente 8 dígitos numéricos): ").strip()

        if cep_input.lower() == 'sair':
            print("Encerrando o programa. Até mais!")
            break

        is_valid, validation_msg = validar_cep(cep_input)
        if not is_valid:
            print(f"⚠️ Atenção: {validation_msg}")
            continue

        print(f"🔍 Consultando CEP '{cep_input}'...")
        resultado = consultar_cep_api(cep_input)
        exibir_resultado(resultado)

# Bloco para garantir que a função main() seja executada apenas quando o script for iniciado diretamente
if __name__ == "__main__":
    main()


Digite o CEP (somente números): 72321010

📍 Resultado:
CEP: 72321-010
Rua: Quadra QR 407 Conjunto 10
Bairro: Samambaia Norte
Cidade: Brasília
Estado: DF
